# MV-DUSt3R+ Run 0 - Sanity Check on Kaggle

Notebook nay clone MV-DUSt3R+ truc tiep tren Kaggle, attach dataset `tiantiansyrinx1102/scannet-data`, download checkpoint tu Hugging Face, va chay sanity check:

```text
input images -> MV-DUSt3R+ -> point cloud/mesh -> scene.glb
```

Day la Run 0, chua tinh metric.

## Before Running

Trong Kaggle Notebook, bat GPU va Add Input dataset:

- Dataset: `tiantiansyrinx1102/scannet-data`
- Internet: On, vi notebook can `git clone` repo va download checkpoint.

In [ ]:
import os, sys, json, time, shutil, subprocess
from pathlib import Path

import torch

print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))

# We want Kaggle GPU T4 x2. Kaggle API exposes only enable_gpu=true,
# so this early check verifies the actual worker assignment.
REQUIRE_T4_X2 = True
if REQUIRE_T4_X2:
    names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    assert torch.cuda.device_count() >= 2 and all('T4' in n for n in names), f'Expected T4 x2, got: {names}'
print('Torch:', torch.__version__)

## Clone MV-DUSt3R+

In [ ]:
REPO_URL = 'https://github.com/facebookresearch/mvdust3r.git'
ROOT = Path('/kaggle/temp/mvdust3r')

if ROOT.exists():
    shutil.rmtree(ROOT)

subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(ROOT)], check=True)
print('Cloned to:', ROOT)

## Install Dependencies

Notebook nay giu lai PyTorch/TorchVision san co cua Kaggle de tranh mismatch CUDA. Cac dependency con lai duoc cai tu `requirements.txt`.

In [ ]:
req_src = ROOT / 'requirements.txt'
req_kaggle = Path('/kaggle/working/requirements_kaggle.txt')

skip_prefixes = ('torch==', 'torchvision==', 'tensorflow', 'gsplat==', 'numpy==', 'scipy', 'opencv-python')
lines = []
for line in req_src.read_text().splitlines():
    s = line.strip()
    if not s or s.startswith('#'):
        continue
    if s.startswith(skip_prefixes):
        print('Skip Kaggle-managed/heavy package:', s)
        continue
    lines.append(s)
req_kaggle.write_text('\n'.join(lines) + '\n')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_kaggle)], check=True)

# Run 0 exports a point cloud GLB and does not need gsplat rendering.
print('Dependency install finished.')

## Locate ScanNet Images

Dataset tu anh cua ban co slug `tiantiansyrinx1102/scannet-data`, thuong duoc mount thanh `/kaggle/input/scannet-data/scannet/posed_images/...`.

In [ ]:
def print_input_tree(max_depth=4, max_items=120):
    root = Path('/kaggle/input')
    print('Input tree preview:')
    if not root.exists():
        print('  /kaggle/input does not exist')
        return
    count = 0
    for p in sorted(root.rglob('*')):
        rel = p.relative_to(root)
        if len(rel.parts) > max_depth:
            continue
        print(' ', rel, '/' if p.is_dir() else '')
        count += 1
        if count >= max_items:
            print('  ...')
            break

def find_posed_images_root():
    candidates = [
        Path('/kaggle/input/scannet-data/scannet/posed_images'),
        Path('/kaggle/input/scannet-data/posed_images'),
        Path('/kaggle/input/tiantiansyrinx1102-scannet-data/scannet/posed_images'),
        Path('/kaggle/input/tiantiansyrinx1102-scannet-data/posed_images'),
        Path('/kaggle/input/scannet/scannet/posed_images'),
        Path('/kaggle/input/scannet/posed_images'),
    ]
    for p in candidates:
        if p.exists():
            return p
    for p in Path('/kaggle/input').rglob('posed_images'):
        if p.exists():
            return p
    print_input_tree()
    raise FileNotFoundError('Cannot find scannet/posed_images under /kaggle/input. Did you Add Input dataset tiantiansyrinx1102/scannet-data?')

POSED_IMAGES = find_posed_images_root()
SCENE = 'scene0000_00'
SCENE_DIR = POSED_IMAGES / SCENE

jpgs = sorted(SCENE_DIR.glob('*.jpg'))
assert len(jpgs) >= 5, f'Need at least 5 jpgs in {SCENE_DIR}, found {len(jpgs)}'

# 5 frames is enough for Run 0. Later runs will control 2/3/4/5 views explicitly.
image_files = [str(p) for p in jpgs[:5]]

print('POSED_IMAGES:', POSED_IMAGES)
print('SCENE_DIR:', SCENE_DIR)
print('Selected images:')
for p in image_files:
    print(' ', p)

## Download Checkpoint

Mac dinh dung `MVDp_s2.pth`, checkpoint MV-DUSt3R+ stage 2.

In [ ]:
from huggingface_hub import hf_hub_download

CKPT_DIR = ROOT / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

ckpt_path = hf_hub_download(
    repo_id='Zhenggang/MV-DUSt3R',
    filename='checkpoints/MVDp_s2.pth',
    local_dir=str(ROOT),
    local_dir_use_symlinks=False,
)

print('Checkpoint:', ckpt_path)

## Run 0 Inference and Export GLB

In [ ]:
import builtins
import types
import numpy as np

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Run 0 does not use PyTorch3D metrics/losses, but dust3r.losses imports
# pytorch3d at module import time. Provide a tiny compatibility shim so
# demo inference can run without compiling PyTorch3D on Kaggle.
try:
    import pytorch3d  # noqa: F401
except ModuleNotFoundError:
    pytorch3d = types.ModuleType('pytorch3d')
    ops = types.ModuleType('pytorch3d.ops')
    transforms = types.ModuleType('pytorch3d.transforms')

    def knn_points(x, y, K=1, **kwargs):
        d = torch.cdist(x, y)
        vals, idx = torch.topk(d, k=K, dim=-1, largest=False)
        return vals, idx, None

    def so3_relative_angle(rot_gt, rot_pred, eps=1e-4, **kwargs):
        r = rot_gt @ rot_pred.transpose(-1, -2)
        trace = r[..., 0, 0] + r[..., 1, 1] + r[..., 2, 2]
        cos = ((trace - 1.0) / 2.0).clamp(-1 + eps, 1 - eps)
        return torch.acos(cos)

    ops.knn_points = knn_points
    transforms.so3_relative_angle = so3_relative_angle
    pytorch3d.ops = ops
    pytorch3d.transforms = transforms
    sys.modules['pytorch3d'] = pytorch3d
    sys.modules['pytorch3d.ops'] = ops
    sys.modules['pytorch3d.transforms'] = transforms

# demo.py has an interactive input() pause inside get_reconstructed_scene.
# Disable it for non-interactive Kaggle execution.
builtins.input = lambda prompt='': ''

from demo import AsymmetricCroCo3DStereoMultiView, get_reconstructed_scene

device = 'cuda' if torch.cuda.is_available() else 'cpu'
inf = np.inf

# Official MV-DUSt3R+ checkpoints store args as Python objects. PyTorch 2.6+
# defaults torch.load(weights_only=True), so use the legacy trusted-checkpoint
# behavior for this official Hugging Face checkpoint.
_torch_load = torch.load
def torch_load_compat(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _torch_load(*args, **kwargs)
torch.load = torch_load_compat

model = AsymmetricCroCo3DStereoMultiView(
    pos_embed='RoPE100', img_size=(224, 224), head_type='linear',
    output_mode='pts3d', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, 1e9),
    enc_embed_dim=1024, enc_depth=24, enc_num_heads=16,
    dec_embed_dim=768, dec_depth=12, dec_num_heads=12,
    GS=True, sh_degree=0, pts_head_config={'skip': True},
    m_ref_flag=True, n_ref=4,
).to(device)

model_loaded = AsymmetricCroCo3DStereoMultiView.from_pretrained(str(ckpt_path)).to(device)
model.load_state_dict(model_loaded.state_dict(), strict=True)
model.eval()

OUT_DIR = Path('/kaggle/working/outputs/run_00_sanity_check')
OUT_DIR.mkdir(parents=True, exist_ok=True)

torch.cuda.empty_cache() if torch.cuda.is_available() else None
start = time.time()

with torch.no_grad():
    output, glb_file, gallery = get_reconstructed_scene(
        str(OUT_DIR),
        model,
        device,
        True,      # silent
        224,       # image_size used by official demo
        image_files,
        3.0,       # confidence threshold percent
        True,      # as_pointcloud
        False,     # transparent_cams
        0.05,      # cam_size
        10,        # n_frame, only relevant for video inputs
    )

runtime = time.time() - start
print('GLB:', glb_file)
print('Runtime seconds:', runtime)

## Save Run Metadata

In [ ]:
run_config = {
    'run': 'run_00_sanity_check',
    'repo_url': REPO_URL,
    'dataset_slug': 'tiantiansyrinx1102/scannet-data',
    'scene': SCENE,
    'image_files': image_files,
    'checkpoint': str(ckpt_path),
    'image_size': 224,
    'confidence_threshold_percent': 3.0,
    'output_glb': str(glb_file),
    'runtime_seconds': runtime,
}

(OUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2))
(OUT_DIR / 'selected_views.json').write_text(json.dumps({'scene': SCENE, 'image_files': image_files}, indent=2))
(OUT_DIR / 'runtime_log.txt').write_text(f'runtime_seconds={runtime}\n')
(OUT_DIR / 'metrics.csv').write_text('run,scene,num_views,runtime_seconds,output_glb\n' + f'run_00_sanity_check,{SCENE},{len(image_files)},{runtime},{glb_file}\n')

print('Saved outputs under:', OUT_DIR)
print('\n'.join(str(p) for p in sorted(OUT_DIR.glob('*'))))

## Expected Output

Sau khi chay xong, tai cac file trong:

```text
/kaggle/working/outputs/run_00_sanity_check/
```

Quan trong nhat la `scene.glb`. Neu file nay visualize duoc thi Run 0 dat muc tieu.